In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
population_weights = df_params[df_params['Type'] == 'population_weights']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Population weights: " + population_weights)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

In [ ]:
## Import Variable Mapping
if estimate == 'DEC':
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = estimate)
else:
    df_vars = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = sample_type)
df_inputs = pd.read_excel(os.path.join(path_config, 'ACS Configuration File.xlsx'), sheet_name = import_tab)
# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set years
years_to_import = list(range(year_start, year_end+1))


## For DEC data
if estimate == 'DEC':

    # Reset years to import for DEC
    # Set DEC variables to import
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    years_to_import = [2000, 2010, 2020]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)

## For ACS1 or ACS5 data
if sample_type == 'ACS':
    
    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    # Set tables and variables to import

    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except:
            pass
    
    list_vars = ['NAME'] + df_vars['ID'].to_list()
    tables = df_vars['Table'].unique()

    # For tract and county level pull
    if import_tab == 'Counties':
        
        # Import County FIPS mapping
        # Convert to dictionary object for easy state-county combination importing
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        
        df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
        dict_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values)) 
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(tables)
        print(list_vars)

    
    # For MSA level pull
    if import_tab == 'MSA':
    
        # Set MSAs to import
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)


## For PUMS data
if import_tab == 'PUMA':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS on these years)
    # Create dictionary of variable mappings by year (sometimes the variable name changes over time)
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except:
            pass
            
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['PUMA'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'PUMAcodes'
                            , dtype = {'State FIPS': object, 'County FIPS': object, 'TRACTCE': object, 'PUMA5CE': object})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values)) 
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])

    # view
    print(dict_fips)
    print(dict_vars)


# view
df_vars.head(3)

In [ ]:
df_table = df_vars[df_vars['Table'] == tables[0]]
list_table_vars = ['NAME']
variables = ",".join(list_table_vars)

year = 2022

df_msa =  query_acs(api_Key     = api_key
                  , estimate  = estimate
                  , sample    = sample_type
                  , geography = geography
                  , variables = variables
                  , year      = year
                  , msa       = '*')

df_msa = df_msa[['metropolitan statistical area/micropolitan statistical area', 'NAME']].rename(
    columns = {'metropolitan statistical area/micropolitan statistical area':'MSA_ID', 'NAME':'MSA'}
)

df_msa.head()

In [ ]:
# Import County FIPS workbook
df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips.head()

In [ ]:
# Extract State Abbreviation from MSA label
def extract_state(text):
    return text.split(',')[1].strip()
    
df_msa['State'] = df_msa['MSA'].apply(extract_state).str[:2]
df_msa.head()

In [ ]:
# Merge State FIPS code onto MSA labels
df_msa2 = df_msa.merge(df_fips[['State', 'State FIPS']].drop_duplicates(), on = 'State', how = 'left')
df_msa2 = df_msa2.sort_values(['MSA_ID']).drop_duplicates()
df_msa2

In [ ]:
with pd.ExcelWriter(os.path.join(path_git, 'config', 'MSA ID Mapping.xlsx'), engine='xlsxwriter') as writer:
            df_msa2.to_excel(writer, index = False, sheet_name = 'MSAcodes')